In [ ]:
!pip -q install prophet xgboost kagglehub

import os, json, time, logging, warnings, zipfile
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from prophet import Prophet
from IPython.display import display

warnings.filterwarnings("ignore")
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
logging.getLogger("prophet").setLevel(logging.WARNING)
np.random.seed(42)

In [ ]:
CATS = ["M01AB", "M01AE", "N02BA", "N02BE", "N05B", "N05C", "R03", "R06"]
NAMES = {
    "M01AB": ("مضادات التهاب (ديكلوفيناك)", "Anti-inflammatory · Diclofenac"),
    "M01AE": ("مضادات التهاب (إيبوبروفين)", "Anti-inflammatory · Ibuprofen"),
    "N02BA": ("مسكنات (أسبرين)", "Analgesics · Aspirin"),
    "N02BE": ("مسكنات (باراسيتامول)", "Analgesics · Paracetamol"),
    "N05B":  ("مضادات القلق", "Anxiolytics"),
    "N05C":  ("المنومات والمهدئات", "Hypnotics & sedatives"),
    "R03":   ("أدوية الربو والجهاز التنفسي", "Obstructive airway drugs"),
    "R06":   ("مضادات الحساسية", "Antihistamines"),
}
MONTHS_AR = ["يناير", "فبراير", "مارس", "أبريل", "مايو", "يونيو",
             "يوليو", "أغسطس", "سبتمبر", "أكتوبر", "نوفمبر", "ديسمبر"]

H_COST = 1.0
P_COST = 4.0
W_COST = 3.0

LEAD_TIME_MONTHS = 1
assert LEAD_TIME_MONTHS == 1, "هذا الإصدار يدعم مدة توريد شهراً واحداً فقط"
SHELF_LIFE = 2

FC_START = 24
TEST_SHARE = 0.2
PLAN_MONTHS = 6
Z_GRID = np.array([-0.5, 0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0])
SAFETY_Z = 1.0

URGENT_DAYS = 7
MONTH_DAYS = 30
OVER_RATIO = 1.3
ALERT_PCT = 20

In [ ]:
def load_data():
    if os.path.exists("salesmonthly.csv"):
        return pd.read_csv("salesmonthly.csv")
    try:
        import kagglehub
        path = kagglehub.dataset_download("milanzdravkovic/pharma-sales-data")
        return pd.read_csv(os.path.join(path, "salesmonthly.csv"))
    except Exception as e:
        print("تعذّر التحميل من Kaggle:", e, "— ارفع ملف salesmonthly.csv يدوياً:")
        from google.colab import files
        up = files.upload()
        return pd.read_csv(list(up.keys())[0])

raw = load_data()
raw["datum"] = pd.to_datetime(raw["datum"])
raw = raw.sort_values("datum").reset_index(drop=True)

total = raw[CATS].sum(axis=1)
q1, q3 = total.quantile([0.25, 0.75])
bad = total < q1 - 1.5 * (q3 - q1)

k = len(raw)
while k > 0 and bad.iloc[k - 1]:
    k -= 1
DROPPED = raw["datum"].iloc[k:].dt.strftime("%Y-%m").tolist()
raw, bad = raw.iloc[:k].copy(), bad.iloc[:k]

IMPUTED = raw.loc[bad, "datum"].dt.strftime("%Y-%m").tolist()
raw.loc[bad, CATS] = np.nan
raw[CATS] = raw[CATS].interpolate(limit_direction="both")

df = raw.reset_index(drop=True)
assert (df["datum"].dt.to_period("M").map(lambda p: p.ordinal).diff().iloc[1:] == 1).all(), "الأشهر غير متصلة"
df["month"] = df["datum"].dt.month
dates = df["datum"]
n = len(df)
split = int(n * (1 - TEST_SHARE))
VAL_START = split - 12
print(f"الأشهر: {n} (حُذف الشهر الناقص {DROPPED}، وعُوِّض الشهر المعطوب {IMPUTED})")
print(f"تدريب: {split} شهراً (منها آخر 12 للتحقق) | اختبار: {n - split} شهراً")

In [ ]:
profile = df.iloc[:split].groupby("month")[CATS].mean()
swing = ((profile.max() - profile.min()) / profile.min() * 100).round(0)
print("الفارق بين أعلى وأدنى شهر (%) لكل فئة:")
display(swing.sort_values(ascending=False).to_frame("swing_%"))

fig, ax = plt.subplots(figsize=(12, 5))
(profile / profile.mean()).plot(ax=ax, marker="o")
ax.axhline(1, color="k", ls="--", lw=0.8)
ax.set_title("Seasonal index by month (1 = category average)")
ax.legend(ncol=4); plt.tight_layout(); plt.show()

train = df.iloc[:split]
STATS = {}
for c in CATS:
    mu = train[c].mean()
    seas = train.groupby("month")[c].mean().reindex(range(1, 13)).fillna(mu).to_dict()
    STATS[c] = dict(mu=mu, seas=seas, sigma=(train[c] - train["month"].map(seas)).std())

In [ ]:
EXT_DATES = list(dates) + [dates.iloc[-1] + pd.offsets.MonthEnd(k) for k in range(1, PLAN_MONTHS + 3)]

def feat_row(hist, date, t):
    m = date.month
    return [np.sin(2*np.pi*m/12), np.cos(2*np.pi*m/12), t,
            hist[-1], hist[-2], hist[-3], hist[-12], np.mean(hist[-3:])]

def xgb_model():
    return XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.05,
                        subsample=0.8, colsample_bytree=0.8, random_state=42)

def xgb_path(y_hist, t0, steps, model):
    hist, out = list(y_hist), []
    for k in range(steps):
        p = max(0.0, float(model.predict(np.array([feat_row(hist, EXT_DATES[t0 + k], t0 + k)]))[0]))
        out.append(p); hist.append(p)
    return out

def prophet_path(y_hist, t0, steps):
    m = Prophet(yearly_seasonality=5, weekly_seasonality=False, daily_seasonality=False, uncertainty_samples=0)
    m.fit(pd.DataFrame({"ds": EXT_DATES[:len(y_hist)], "y": y_hist}))
    return list(np.clip(m.predict(pd.DataFrame({"ds": EXT_DATES[t0:t0 + steps]}))["yhat"].values, 0, None))

def walk_forward(y):
    N = len(y)
    fx1, fx2, fp1, fp2 = (np.full(N + 2, np.nan) for _ in range(4))
    for o in range(FC_START - 1, N):
        X = np.array([feat_row(y[:k], EXT_DATES[k], k) for k in range(12, o + 1)])
        m = xgb_model().fit(X, y[12:o + 1])
        fx1[o + 1], fx2[o + 2] = xgb_path(y[:o + 1], o + 1, 2, m)
        fp1[o + 1], fp2[o + 2] = prophet_path(y[:o + 1], o + 1, 2)
    return {"XGBoost": (fx1, fx2), "Prophet": (fp1, fp2), "Average": ((fx1 + fp1) / 2, (fx2 + fp2) / 2)}

def rel_err(y, f, s):
    return np.mean(np.abs(y[s] - f[s])) / y[s].mean() * 100

t0 = time.time()
FC1, FC2, FRES, SIG_POS, BEST_MODEL, FC_ERR = {}, {}, {}, {}, {}, {}
val, tst = slice(VAL_START, split), slice(split, n)
for c in CATS:
    y = df[c].values.astype(float)
    cands = walk_forward(y)
    err = {k: {"val": rel_err(y, f1, val), "test": rel_err(y, f1, tst)} for k, (f1, f2) in cands.items()}
    BEST_MODEL[c] = min(err, key=lambda k: err[k]["val"])
    FC_ERR[c] = err
    FC1[c], FC2[c] = cands[BEST_MODEL[c]]
    FRES[c] = (y - FC1[c][:n])[FC_START:split]
    idx = np.arange(FC_START + 1, split - 1)
    SIG_POS[c] = ((y[idx] + y[idx + 1]) - (FC1[c][idx] + FC2[c][idx + 1])).std()
    print(f"✓ {c} | النموذج المختار: {BEST_MODEL[c]}")

display(pd.DataFrame({c: {**{f"{k} (اختبار %)": round(v["test"], 1) for k, v in FC_ERR[c].items()},
                          "المختار": BEST_MODEL[c]} for c in CATS}).T)
print(f"انتهت التوقعات في {time.time() - t0:.0f} ثانية")

In [ ]:
class PharmacyEnv:
    def __init__(self, c, demand=None):
        self.c = c
        self.d = df[c].values.astype(float) if demand is None else np.asarray(demand, float)
        self.m = df["month"].values
        self.f1, self.f2 = FC1[c], FC2[c]

    def reset(self, start, end, inv0=None):
        self.t, self.end = start, end
        q0 = self.f1[start] if inv0 is None else inv0
        self.batches = [[q0, 0]] if q0 > 0 else []
        self.last = self.d[start - 1]
        return self.state()

    def on_hand(self):
        return sum(q for q, a in self.batches)

    def state(self):
        t = self.t
        return dict(on_hand=self.on_hand(),
                    expiring=sum(q for q, a in self.batches if a == SHELF_LIFE - 1),
                    fc0=self.f1[t], fc1=self.f2[t + 1], last=self.last, month=int(self.m[t]))

    def step(self, q):
        q = max(0.0, float(q))
        dem = self.d[self.t]
        need = dem
        self.batches.sort(key=lambda b: -b[1])
        for b in self.batches:
            take = min(b[0], need); b[0] -= take; need -= take
        sales = dem - need
        for b in self.batches:
            b[1] += 1
        expired = sum(q_ for q_, a in self.batches if a >= SHELF_LIFE)
        self.batches = [b for b in self.batches if b[1] < SHELF_LIFE and b[0] > 1e-9]
        left = self.on_hand()
        cost = H_COST * left + P_COST * need + W_COST * expired
        if q > 0:
            self.batches.append([q, 0])
        self.last = sales
        self.t += 1
        done = self.t >= self.end
        info = dict(order=q, demand=dem, sales=sales, lost=need, expired=expired, inv=left, cost=cost)
        return (None if done else self.state()), -cost, done, info

def run_policy(env, policy, start, end, inv0=None):
    s = env.reset(start, end, inv0)
    logs = []
    while True:
        s2, r, done, info = env.step(policy(s))
        logs.append(info)
        if done: break
        s = s2
    return pd.DataFrame(logs)

def leftover(on_hand, expiring, fc0):
    return max(0.0, on_hand - max(fc0, expiring))

def order_qty(fc0, fc1, on_hand, expiring, safety):
    return max(0.0, fc1 + safety - leftover(on_hand, expiring, fc0))

def rule(c, z):
    return lambda s: order_qty(s["fc0"], s["fc1"], s["on_hand"], s["expiring"], z * SIG_POS[c])

def traditional(c):
    st = STATS[c]; seas = st["seas"]; sig2 = st["sigma"] * np.sqrt(2)
    nxt = lambda m: seas[m % 12 + 1]
    return {
        "Fixed order":       lambda s: max(0.0, 2 * st["mu"] - s["on_hand"]),
        "Last month":        lambda s: max(0.0, 2 * s["last"] - s["on_hand"]),
        "Seasonal mean":     lambda s: max(0.0, seas[s["month"]] + nxt(s["month"]) - s["on_hand"]),
        "Seasonal + safety": lambda s: max(0.0, seas[s["month"]] + nxt(s["month"]) + SAFETY_Z * sig2 - s["on_hand"]),
        "Forecast + safety": rule(c, SAFETY_Z),
    }

MAIN, CURRENT = "System", "Seasonal mean"
inv0_of = lambda c: FC1[c][split]
LOGS, rows, TUNED_Z = {c: {} for c in CATS}, [], {}
for c in CATS:
    env = PharmacyEnv(c)
    val_costs = {z: run_policy(env, rule(c, z), VAL_START, split).cost.sum() for z in Z_GRID}
    TUNED_Z[c] = float(min(val_costs, key=val_costs.get))
    P = traditional(c)
    P[MAIN] = rule(c, TUNED_Z[c])
    for name, pol in P.items():
        log = run_policy(env, pol, split, n, inv0_of(c))
        LOGS[c][name] = log
        rows.append({"category": c, "policy": name, "total_cost": log.cost.sum(), "lost_units": log.lost.sum(),
                     "expired_units": log.expired.sum(), "avg_inventory": log.inv.mean(),
                     "sales": log.sales.sum(), "demand": log.demand.sum()})
    print(f"✓ {c} | الاحتياطي المضبوط: ×{TUNED_Z[c]}")

results = pd.DataFrame(rows)
overall = results.groupby("policy")[["total_cost", "lost_units", "expired_units", "sales", "demand"]].sum()
overall["avg_inventory"] = results.groupby("policy")["avg_inventory"].mean()
overall["fill_rate_%"] = overall.sales / overall.demand * 100
overall = overall.drop(columns=["sales", "demand"]).sort_values("total_cost").round(1)
display(overall)

cur, sysr = overall.loc[CURRENT], overall.loc[MAIN]
print(f"\n===== النتائج على {n - split} شهراً لم يرها النظام =====")
print(f"التوفير مقارنة بالطريقة الحالية: {(1 - sysr.total_cost / cur.total_cost) * 100:.1f}%")
print(f"المبيعات الضائعة: {cur.lost_units:.0f} ← {sysr.lost_units:.0f}")
print(f"العلب التالفة:    {cur.expired_units:.0f} ← {sysr.expired_units:.0f}")
print(f"نسبة تلبية الطلب: {cur['fill_rate_%']:.1f}% ← {sysr['fill_rate_%']:.1f}%")

In [ ]:
H = PLAN_MONTHS + 1

def future_forecast(c):
    y = df[c].values.astype(float)
    X = np.array([feat_row(y[:k], EXT_DATES[k], k) for k in range(12, n)])
    fx = xgb_path(y, n, H, xgb_model().fit(X, y[12:]))
    fp = prophet_path(y, n, H)
    return {"XGBoost": fx, "Prophet": fp, "Average": list((np.array(fx) + np.array(fp)) / 2)}[BEST_MODEL[c]]

def current_batches(c):
    env = PharmacyEnv(c)
    s = env.reset(split, n, inv0_of(c))
    pol = rule(c, TUNED_Z[c])
    while True:
        s2, _, done, _ = env.step(pol(s))
        if done: break
        s = s2
    return [list(b) for b in env.batches], env.last

def month_reason(pct, prev_fc, fc, expiring):
    if pct >= ALERT_PCT:    txt = f"ذروة موسمية: الطلب في هذا الشهر عادةً أعلى من متوسط السنة بـ {round(pct)}%."
    elif pct <= -ALERT_PCT: txt = f"ركود موسمي: الطلب في هذا الشهر عادةً أقل من متوسط السنة بـ {round(-pct)}%."
    else:                   txt = "الطلب في هذا الشهر عادةً قريب من متوسط السنة."
    if prev_fc:
        ch = (fc / prev_fc - 1) * 100
        if abs(ch) >= 10:
            txt += f" {'أعلى' if ch > 0 else 'أقل'} من الشهر السابق بـ {round(abs(ch))}%."
    if expiring >= 1:
        txt += f" تنتهي صلاحية {round(expiring)} علبة نهاية الشهر."
    return txt

SEASON_PCT = {c: {int(m): (v / df[c].mean() - 1) * 100 for m, v in df.groupby("month")[c].mean().items()} for c in CATS}

rnd = lambda v: int(np.floor(v + 0.5))

PLAN, NOW = {}, {}
for c in CATS:
    y = df[c].values.astype(float)
    fut = future_forecast(c)
    batches, last = current_batches(c)
    NOW[c] = dict(batches=[list(b) for b in batches], last=last)
    plan, prev_fc = [], float(y[-1])
    for k in range(PLAN_MONTHS):
        date = EXT_DATES[n + k]
        on_hand = sum(q for q, a in batches)
        expiring = sum(q for q, a in batches if a == SHELF_LIFE - 1)
        fc0, fc1 = fut[k], fut[k + 1]
        parts = {"need_arrival": rnd(fc1), "safety": rnd(TUNED_Z[c] * SIG_POS[c]), "leftover": rnd(leftover(on_hand, expiring, fc0))}
        order = max(0, parts["need_arrival"] + parts["safety"] - parts["leftover"])
        pct = SEASON_PCT[c].get(date.month, 0.0)
        same = df.loc[df["month"] == date.month, ["datum", c]]
        plan.append({
            "month": date.strftime("%Y-%m"),
            "forecast": round(fc0, 1),
            "on_hand_start": round(on_hand),
            "expiring": round(expiring),
            "order": order,
            "order_parts": parts,
            "arrives": EXT_DATES[n + k + 1].strftime("%Y-%m"),
            "season_pct": round(pct, 1),
            "same_month_history": [{"year": int(d.year), "sales": round(float(v), 1)} for d, v in same.values],
            "reason": month_reason(pct, prev_fc, fc0, expiring),
        })
        need = fc0
        batches.sort(key=lambda b: -b[1])
        for b in batches:
            take = min(b[0], need); b[0] -= take; need -= take
        for b in batches:
            b[1] += 1
        batches = [b for b in batches if b[1] < SHELF_LIFE and b[0] > 1e-9]
        if order > 0:
            batches.append([order, 0])
        prev_fc = fc0
    PLAN[c] = plan
    print(f"✓ {c}: " + " | ".join(f"{p['month']}: {p['order']}" for p in plan))

In [ ]:
STATUS_INFO = {
    "out":    ("نفد", "لا توجد أي علبة في المخزون"),
    "urgent": ("ينفد هذا الأسبوع", "المخزون يكفي أقل من أسبوع"),
    "order":  ("ينفد هذا الشهر", "المخزون يكفي أسبوعاً أو أكثر، لكنه لا يكفي الشهر كاملاً"),
    "over":   ("مخزون زائد", "المخزون أعلى من حاجة الشهرين القادمين بـ 30% أو أكثر"),
    "ok":     ("يكفي هذا الشهر", "المخزون يكفي الشهر كاملاً"),
}

def status_of(inv, cover_days, need2):
    if inv < 1:                   return "out"
    if cover_days < URGENT_DAYS:  return "urgent"
    if inv > OVER_RATIO * need2:  return "over"
    if cover_days < MONTH_DAYS:   return "order"
    return "ok"

ym = lambda d: pd.Timestamp(d).strftime("%Y-%m")
r1 = lambda a: [round(float(v), 1) for v in a]

def reco_explain(c, p0, p1):
    m0, m1 = MONTHS_AR[int(p0["month"][5:]) - 1], MONTHS_AR[int(p1["month"][5:]) - 1]
    inv, exp_ = p0["on_hand_start"], p0["expiring"]
    need1, safety, left = p0["order_parts"]["need_arrival"], p0["order_parts"]["safety"], p0["order_parts"]["leftover"]
    order = p0["order"]
    if left == 0:
        first = f"المخزون الحالي ({inv}) يُستهلك في {m0}، فلا يتبقى منه شيء لـ{m1}."
    else:
        first = f"يتبقى من المخزون الحالي ({inv}) حوالي {left} بعد استهلاك {m0}."
    if exp_ > p0["forecast"]:
        first += f" منها {round(exp_ - p0['forecast'])} علبة تنتهي صلاحيتها قبل أن تُصرف ولا تُحسب."
    if order == 0:
        return f"{first} المتبقي يكفي طلب {m1} ({need1}) والاحتياطي ({safety})، فلا حاجة للطلب."
    return (f"{first} الكمية = طلب {m1} ({need1}) + احتياطي ({safety}) − المتبقي ({left}) "
            f"= {order}.")

cats_out, alerts = [], []
for c in CATS:
    y = df[c].values.astype(float)
    p0 = PLAN[c][0]
    inv, fc0, fc1 = p0["on_hand_start"], p0["forecast"], PLAN[c][1]["forecast"]
    cover = inv / max(fc0, 1e-6) * 30
    status = status_of(inv, cover, fc0 + fc1)
    log = LOGS[c][MAIN]
    dem, lost = float(log.demand.sum()), float(log.lost.sum())

    near = PLAN[c][:3]
    up = max(near, key=lambda p: p["season_pct"]); dn = min(near, key=lambda p: p["season_pct"])
    for p, kind in ((up, "up"), (dn, "down")):
        if (kind == "up" and p["season_pct"] >= ALERT_PCT and p["season_pct"] >= -dn["season_pct"]) or \
           (kind == "down" and p["season_pct"] <= -ALERT_PCT and -p["season_pct"] > up["season_pct"]):
            pct = round(abs(p["season_pct"]))
            alerts.append({"code": c, "type": kind, "pct": pct,
                           "title": f"{NAMES[c][0]}: {'ذروة' if kind == 'up' else 'ركود'} في {MONTHS_AR[int(p['month'][5:]) - 1]}",
                           "text": (f"الطلب في هذا الشهر عادةً أعلى من متوسط السنة بـ {pct}% · يُنصح بتجهيز مخزون كافٍ" if kind == "up"
                                    else f"الطلب في هذا الشهر عادةً أقل من متوسط السنة بـ {pct}% · يُنصح بتخفيف الطلب لتجنّب التكدّس")})

    cats_out.append({
        "code": c, "name_ar": NAMES[c][0], "name_en": NAMES[c][1], "model": BEST_MODEL[c],
        "inventory": inv, "expiring": p0["expiring"], "last_sales": round(NOW[c]["last"], 1),
        "forecast_next": fc0, "forecast_arrival": fc1,
        "safety_added": round(TUNED_Z[c] * SIG_POS[c]), "recommended_order": p0["order"],
        "cover_days": round(cover, 1),
        "status": status, "status_label": STATUS_INFO[status][0], "status_hint": STATUS_INFO[status][1],
        "reason": p0["reason"],
        "leftover_after": p0["order_parts"]["leftover"],
        "reco_explain": reco_explain(c, p0, PLAN[c][1]),
        "outlook_change": round((np.mean([p["forecast"] for p in PLAN[c][:2]]) / np.mean(y[-3:]) - 1) * 100, 1),
        "plan": PLAN[c],
        "policies": [{"policy": r.policy, "total_cost": round(float(r.total_cost), 1), "lost_units": round(float(r.lost_units), 1),
                      "expired_units": round(float(r.expired_units), 1), "fill_rate_%": round(float(r.sales / r.demand * 100), 1)}
                     for r in results[results.category == c].sort_values("total_cost").itertuples()],
        "perf": {"fill_rate": round((1 - lost / dem) * 100 if dem else 100.0, 1), "lost": round(lost, 1),
                 "expired": round(float(log.expired.sum()), 1), "avg_inventory": round(float(log.inv.mean()), 1),
                 "forecast_accuracy": round(100 - FC_ERR[c][BEST_MODEL[c]]["test"], 1)},
        "history": {"months": [ym(d) for d in dates], "sales": r1(y)},
        "forecast2": {"months": [PLAN[c][0]["month"], PLAN[c][1]["month"]], "values": [fc0, fc1]},
        "seasonal_index": [round(1 + SEASON_PCT[c].get(m, 0.0) / 100, 3) for m in range(1, 13)],
        "test": {"months": [ym(d) for d in dates[split:]], "demand": r1(log.demand), "order": r1(log.order),
                 "inv": r1(log.inv), "lost": r1(log.lost), "expired": r1(log.expired), "fc": r1(FC1[c][split:n])},
    })

alerts = sorted(alerts, key=lambda a: -a["pct"])[:4]
RANK = {"out": 0, "urgent": 1, "order": 2, "over": 3, "ok": 4}
cats_out.sort(key=lambda x: (RANK[x["status"]], x["cover_days"]))
counts = {k: sum(1 for x in cats_out if x["status"] == k) for k in STATUS_INFO}
sysr, cur = overall.loc[MAIN], overall.loc[CURRENT]

data = {
    "meta": {"generated": datetime.now().strftime("%Y-%m-%d %H:%M"),
             "current_month": ym(dates.iloc[-1]), "next_month": PLAN[CATS[0]][0]["month"],
             "arrival_month": PLAN[CATS[0]][0]["arrives"], "first_month": ym(dates.iloc[0]),
             "test_months": int(n - split), "dropped_months": DROPPED, "imputed_months": IMPUTED, "plan_months": PLAN_MONTHS,
             "lead_time_months": LEAD_TIME_MONTHS, "shelf_life_months": SHELF_LIFE,
             "costs": {"holding": H_COST, "stockout": P_COST, "expired": W_COST},
             "display_year_shift": max(0, datetime.now().year - int(ym(dates.iloc[-1])[:4]))},
    "status_info": {k: {"label": v[0], "hint": v[1]} for k, v in STATUS_INFO.items()},
    "summary": {
        "fill_rate": round(float(sysr["fill_rate_%"]), 1), "lost_units": round(float(sysr.lost_units), 1),
        "expired_units": round(float(sysr.expired_units), 1), "avg_inventory": round(float(sysr.avg_inventory), 1),
        "forecast_accuracy": round(float(np.mean([x["perf"]["forecast_accuracy"] for x in cats_out])), 1),
        "total_order_units": int(sum(x["recommended_order"] for x in cats_out)),
        "n_need_order": sum(1 for x in cats_out if x["recommended_order"] > 0),
        "n_critical": counts["out"] + counts["urgent"], "counts": counts,
    },
    "benchmark": {
        "current_method": CURRENT,
        "saving_pct": round((1 - sysr.total_cost / cur.total_cost) * 100, 1),
        "policies": [{"policy": p, **{k: round(float(v), 1) for k, v in r.items()}} for p, r in overall.iterrows()],
    },
    "alerts": alerts,
    "categories": cats_out,
}

sales_long = df.melt(id_vars=["datum"], value_vars=CATS, var_name="code", value_name="sales")
sales_long["month"] = sales_long["datum"].dt.strftime("%Y-%m")
sales_long["year"] = sales_long["datum"].dt.year
sales_long["month_num"] = sales_long["datum"].dt.month
sales_long["name_ar"] = sales_long["code"].map(lambda c: NAMES[c][0])
sales_long["imputed"] = sales_long["month"].isin(IMPUTED).astype(int)
sales_long = sales_long[["month", "year", "month_num", "code", "name_ar", "sales", "imputed"]]

with open("data.js", "w", encoding="utf-8") as f:
    f.write("window.DASH_DATA = " + json.dumps(data, ensure_ascii=False, indent=1) + ";\n")
with open("data.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=1)
sales_long.to_csv("sales.csv", index=False, encoding="utf-8")

with zipfile.ZipFile("pharmastock_export.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for fn in ("data.js", "data.json", "sales.csv"):
        z.write(fn)
print("تم التصدير ✓ — pharmastock_export.zip")
try:
    from google.colab import files
    files.download("pharmastock_export.zip")
except ImportError:
    pass